# 07 — Biomarker Discovery

We integrate signals from:
- Differential expression (t-test / Mann–Whitney)
- PCA gene loadings on discriminative PCs
- NMF metagene gene weights
- Network hub centrality

to derive a ranked list of candidate disease biomarkers.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = '/content/gene-expression-la-analysis'
sys.path.insert(0, f'{PROJECT_ROOT}/src')

from validation import t_test_de, mann_whitney_de, hypergeometric_enrichment
from visualization import plot_volcano, plot_expression_heatmap, plot_gene_loadings
from linear_algebra import get_top_genes_by_loading

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
expr     = pd.read_csv(f'{PROJECT_ROOT}/data/processed/processed_expression.csv',
                        index_col=0)
meta     = pd.read_csv(f'{PROJECT_ROOT}/data/processed/metadata_clean.csv',
                        index_col=0)
pca_load = pd.read_csv(f'{PROJECT_ROOT}/data/processed/pca_loadings.csv',
                        index_col=0)

try:
    nmf_W = pd.read_csv(f'{PROJECT_ROOT}/data/processed/nmf_W.csv', index_col=0)
except FileNotFoundError:
    nmf_W = None
    print('NMF W not found — run notebook 04 first')

try:
    hubs = pd.read_csv(f'{PROJECT_ROOT}/data/results/05_hub_genes.csv',
                        index_col=0, header=None, names=['degree'])
except FileNotFoundError:
    hubs = None

LABEL_COL = 'source_name_ch1'   # <-- adjust
if LABEL_COL not in meta.columns:
    np.random.seed(1)
    meta['group'] = np.random.choice(['Disease','Control'], size=len(meta))
    LABEL_COL = 'group'

common = expr.columns.intersection(meta.index)
expr   = expr[common]
meta   = meta.loc[common]
labels = meta[LABEL_COL].dropna()

classes = labels.unique()
GROUP_A, GROUP_B = classes[0], classes[1] if len(classes) > 1 else (classes[0], classes[0])
print(f'Comparing: {GROUP_A} vs {GROUP_B}')


## 1. Differential Expression

In [ ]:
de_t = t_test_de(expr, labels, group_a=GROUP_A, group_b=GROUP_B)
print(de_t.head(10))


In [ ]:
plot_volcano(de_t, fc_thresh=1.0, p_thresh=0.05,
             title=f'Volcano — {GROUP_A} vs {GROUP_B}',
             save_path=f'{PROJECT_ROOT}/data/results/07_volcano.png')


In [ ]:
# Significant DE genes
sig_de = de_t[(de_t['padj'] < 0.05) & (de_t['log2FoldChange'].abs() >= 1.0)]
de_up   = sig_de[sig_de['log2FoldChange'] > 0].index.tolist()
de_down = sig_de[sig_de['log2FoldChange'] < 0].index.tolist()
print(f'DE genes: {len(sig_de)} ({len(de_up)} up, {len(de_down)} down)')


## 2. PCA Loading–Based Gene Scores

In [ ]:
# Use top discriminative PCs from notebook 06 (or default PC1-5)
try:
    pc_imp = pd.read_csv(f'{PROJECT_ROOT}/data/results/06_pc_importances.csv',
                          index_col=0, header=None, names=['importance'])
    top_pcs = pc_imp.head(5).index.tolist()
except FileNotFoundError:
    top_pcs = ['PC1','PC2','PC3']

print(f'Discriminative PCs: {top_pcs}')

# Aggregate: sum of |loading| across top PCs
pca_score = pca_load[top_pcs].abs().sum(axis=1)
pca_score.name = 'pca_aggregate_loading'
pca_score = pca_score.sort_values(ascending=False)
print(f'Top 10 genes by aggregate PCA loading:')
print(pca_score.head(10))


## 3. NMF Metagene Weight–Based Scores

In [ ]:
if nmf_W is not None:
    nmf_score = nmf_W.max(axis=1)    # max weight across all metagenes
    nmf_score.name = 'nmf_max_weight'
    nmf_score = nmf_score.sort_values(ascending=False)
    print('Top 10 genes by NMF weight:')
    print(nmf_score.head(10))
else:
    nmf_score = pd.Series(dtype=float, name='nmf_max_weight')


## 4. Integrate Signals — Biomarker Ranking

In [ ]:
# Build a unified ranking table
all_genes = expr.index.tolist()
rank_df   = pd.DataFrame(index=all_genes)

# DE score: -log10(padj) × sign(log2FC)
de_t_align = de_t.reindex(all_genes)
rank_df['de_score']  = (-np.log10(de_t_align['padj'].clip(lower=1e-300)) *
                         np.sign(de_t_align['log2FoldChange']))
rank_df['de_abs']    = rank_df['de_score'].abs()
rank_df['pca_score'] = pca_score.reindex(all_genes)

if not nmf_score.empty:
    rank_df['nmf_score'] = nmf_score.reindex(all_genes)
else:
    rank_df['nmf_score'] = np.nan

if hubs is not None:
    rank_df['hub_degree'] = hubs['degree'].reindex(all_genes)
else:
    rank_df['hub_degree'] = np.nan

# Min-max normalise each score to [0,1]
score_cols = ['de_abs', 'pca_score', 'nmf_score', 'hub_degree']
for col in score_cols:
    s = rank_df[col].fillna(0)
    mn, mx = s.min(), s.max()
    rank_df[col+'_norm'] = (s - mn) / (mx - mn + 1e-12)

norm_cols = [c+'_norm' for c in score_cols]
rank_df['composite_score'] = rank_df[norm_cols].mean(axis=1)
rank_df = rank_df.sort_values('composite_score', ascending=False)

print('Top 20 candidate biomarkers:')
print(rank_df[['de_abs','pca_score','nmf_score','hub_degree','composite_score']]
      .head(20).round(4))


In [ ]:
# ── Visualise top biomarkers ───────────────────────────────────────────────
top_biomarkers = rank_df.head(30).index.tolist()

top_bio_common = [g for g in top_biomarkers if g in expr.index]
plot_expression_heatmap(
    expr.loc[top_bio_common],
    col_labels=labels.reindex(expr.columns),
    top_n_genes=len(top_bio_common),
    title='Top 30 Biomarker Candidates — Expression Heatmap',
    save_path=f'{PROJECT_ROOT}/data/results/07_biomarker_heatmap.png'
)


In [ ]:
# ── Composite score bar chart ──────────────────────────────────────────────
top20 = rank_df.head(20)

fig, ax = plt.subplots(figsize=(8, 7))
top20['composite_score'][::-1].plot.barh(color='teal', alpha=0.8, ax=ax)
ax.set_xlabel('Composite Biomarker Score')
ax.set_title('Top 20 Candidate Biomarkers (Integrated Ranking)')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/07_biomarker_ranking.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Boxplots of top 6 biomarkers ──────────────────────────────────────────
top6 = top_biomarkers[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, gene in zip(axes.flat, top6):
    if gene not in expr.index:
        continue
    plot_data = pd.DataFrame({
        'Expression': expr.loc[gene, common],
        'Group':      labels.reindex(common)
    })
    sns.boxplot(data=plot_data, x='Group', y='Expression',
                palette='Set2', ax=ax)
    ax.set_title(gene, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Top 6 Biomarker Candidates — Expression by Group', fontsize=13)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/data/results/07_biomarker_boxplots.png',
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Save final results ────────────────────────────────────────────────────
rank_df.to_csv(f'{PROJECT_ROOT}/data/results/07_biomarker_ranking.csv')
de_t.to_csv(f'{PROJECT_ROOT}/data/results/07_differential_expression.csv')

print('\n── Analysis Complete ──')
print(f'Top 10 candidate biomarkers:')
print(rank_df.index[:10].tolist())
print(f'\nResults saved to: {PROJECT_ROOT}/data/results/')


## Summary Table

| Step | Output file |
|------|-------------|
| Differential expression | `07_differential_expression.csv` |
| Composite biomarker ranking | `07_biomarker_ranking.csv` |
| Volcano plot | `07_volcano.png` |
| Biomarker heatmap | `07_biomarker_heatmap.png` |
| Ranking bar chart | `07_biomarker_ranking.png` |
| Top-6 boxplots | `07_biomarker_boxplots.png` |